In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
import sys
sys.path.append('../Datamaneger')

from Datamaneger import Data_manager

sys.path.append('../Data_Transformer')
from Datatransformer import Data_transformer



In [10]:
class K_fold:
    def __init__(self,df):
        self.df=df
        self.all=[]
        self.cnt=0

    def fold(self):
        #X=self.df.sort_values("tick")
        #X=X.drop(columns=["tick","ticker"])#maybe have to group by ticker or similar, modify here
        #Y=datatransformer
        K_fold=TimeSeriesSplit(n_splits=5,gap=10)

        for _,(train_index,test_index) in enumerate(K_fold.split(self.df)):
            X_train = X.iloc[train_index[0] : train_index[-1] + 1]
            X_val  = X.iloc[test_index[0] : test_index[-1] + 1]

            #Y_train = Y.iloc[train_index[0] : train_index[0] + cut]
            #Y_val   = Y.iloc[test_index[0] : test_index[-1] + 1]

            scaler = StandardScaler()
            scaler.fit(X_train)

            X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns)
            X_val_scaled   = pd.DataFrame(scaler.transform(X_val),   columns=X_train.columns)

            self.all.append([X_train_scaled,X_val_scaled])

    def get_data(self):
        self.cnt+=1
        self.cnt%=len(self.all)
        return self.all[self.cnt]



In [11]:
import yfinance as yf

mydf = yf.download('AAPL', period='2y')

mydf.columns = mydf.columns.get_level_values(0) 
mydf['Ticker'] = 'AAPL'   

Manager=Data_manager(dataf=mydf)
temp=Manager.get_traindata()
#print(temp.head())
Transformer=Data_transformer(temp)
#print(Transformer.df.head())
Transformer.transform()
#print(Transformer.df.head())
X=Transformer.get_X()
Y=Transformer.get_Y()
tid=Transformer.get_time_since_last_market_day()

print(X.head())
print(Y.head())
#print(tid)

A=K_fold(X)
A.fold()

for i in range(10):
    B,C=A.get_data()
    print(B.shape,C.shape)

[*********************100%***********************]  1 of 1 completed

Price             rt      rt_3      rt_5     rt_10     rt_20   sigma_5  \
Date                                                                     
2024-05-10 -0.006914  0.004913  0.057650  0.093720  0.082282  0.021772   
2024-05-13  0.017492  0.020542  0.026194  0.093455  0.101967  0.009133   
2024-05-14  0.006155  0.016732  0.028559  0.103089  0.100906  0.009100   
2024-05-16  0.012776  0.036422  0.039473  0.109799  0.124883  0.009254   
2024-05-17  0.000158  0.019089  0.029666  0.116023  0.082681  0.009732   

Price       sigma_10  sigma_20        RV       rho        vz  delta_vwap  \
Date                                                                       
2024-05-10  0.016860  0.018899  0.000260  1.151979 -0.535585    0.052768   
2024-05-13  0.016845  0.019065  0.000178  0.479037  0.510192    0.064388   
2024-05-14  0.016297  0.019061  0.000115  0.477444 -0.603226    0.066354   
2024-05-16  0.016241  0.018736  0.000057  0.493900 -0.590409    0.073406   
2024-05-17  0.015623  0.0